# ASTA - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains ASTA on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: ASTA (DehazeFormer variant)  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [1]:
#@title 1. Install Dependencies
!pip install timm pytorch-msssim lpips scikit-image thop tensorboardX opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.0 MB/s eta 0:00:00


In [3]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/ASTA_Results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/weights"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

--2026-06-22 00:26:19--  https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.4.18, 2620:100:6017:18::a27d:212
Connecting to www.dropbox.com (www.dropbox.com)|162.125.4.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1 [following]
--2026-06-22 00:26:19--  https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucc5d62ed4eb2192b6db35aa37bc.dl.dropboxusercontent.com/cd/0/inline/DC3eMDFTwpES3i7j9yoqAM2djBwKZZu39n9MTqwjbGkw7iQBFr1WlqqSXkphS3gN5C60doiul621InZD3wkyOkMAM6VJPbbLGdUJhepB8MWfj9DfgUrb-TM2oY4UJiy93Ny4AhyANIq-zzibqOjRQUM7/file?dl=1# [following]
--2026-06-22 00:26:20--  https://ucc5d62ed4eb2192b6db35aa37bc.dl.dropboxusercontent.com

In [5]:
#@title 4. Clone ASTA Repository
!git clone https://github.com/sumire25/ASTA.git
%cd ASTA
!ls

Cloning into 'ASTA'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 30 (delta 5), reused 30 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 17.40 MiB | 17.68 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/ASTA
 ASTA.ipynb    evaluate.py  'model weights'     run.sh	   utils
 config.json   infer.py      README.md	        test.py
 datasets      models	     requirements.txt   train.py


In [6]:
#@title 5. Prepare Dataset Structure for ASTA
import os
import shutil

src_base = '/content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset'
dst_base = './data/Haze1k-thick'

os.makedirs(f'{dst_base}/train/input', exist_ok=True)
os.makedirs(f'{dst_base}/train/target', exist_ok=True)
os.makedirs(f'{dst_base}/test/input', exist_ok=True)
os.makedirs(f'{dst_base}/test/target', exist_ok=True)

for split in ['train', 'valid', 'test']:
    dst_split = 'test' if split == 'valid' else split
    for sub in ['input', 'target']:
        src_dir = os.path.join(src_base, split, sub)
        dst_dir = os.path.join(dst_base, dst_split, sub)
        if os.path.exists(src_dir):
            for f in os.listdir(src_dir):
                src_path = os.path.join(src_dir, f)
                dst_path = os.path.join(dst_dir, f)
                if not os.path.exists(dst_path):
                    shutil.copy2(src_path, dst_path)

print('Dataset prepared!')
!ls -l ./data/Haze1k-thick/train/input/ | head -5
!ls -l ./data/Haze1k-thick/test/input/ | head -5

Dataset prepared!
total 90380
-rw-r--r-- 1 root root 299118 Oct 11  2019 100-inputs.png
-rw-r--r-- 1 root root 248336 Oct 11  2019 101-inputs.png
-rw-r--r-- 1 root root 282777 Oct 11  2019 102-inputs.png
-rw-r--r-- 1 root root 279127 Oct 11  2019 103-inputs.png
total 24740
-rw-r--r-- 1 root root 347863 Oct 11  2019 321-inputs.png
-rw-r--r-- 1 root root 296328 Oct 11  2019 322-inputs.png
-rw-r--r-- 1 root root 310179 Oct 11  2019 323-inputs.png
-rw-r--r-- 1 root root 305822 Oct 11  2019 324-inputs.png


In [7]:
#@title 6. Compute FLOPs and Parameters
import torch
from thop import profile
from models import asta

network = asta().cuda()
network.eval()

dummy_input = torch.randn(1, 3, 256, 256).cuda()
macs, params = profile(network, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- ASTA Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register count_softmax() for <class 'torch.nn.modules.activation.Softmax'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.pixelshuffle.PixelShuffle'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.

--- ASTA Complexity ---
FLOPs: 39.0371 G
Parameters: 3.3224 M


## Phase 1: Test Pretrained Weights

In [8]:
#@title 7. Run Inference with Pretrained Weights
!python infer.py \
  --model asta \
  --weights "model weights/saved_models_SateHaze1k-TK/asta.pth" \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --result_dir /content/drive/MyDrive/ASTA_Results/pretrained_results/

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
==> Loading weights: model weights/saved_models_SateHaze1k-TK/asta.pth
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader 

In [9]:
#@title 8. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/ASTA_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
100% 233M/233M [00:01<00:00, 162MB/s]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 3.014
ERGAS: 15.343
UIQI: 0.845
QNR: 

## Phase 2: Train from Scratch

In [11]:
#@title 9. Train ASTA
!python train.py \
  --model asta \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --save_dir /content/drive/MyDrive/ASTA_Results/weights/ \
  --log_dir ./logs/ \
  --num_workers 4 \
  --val_freq 3 \
  --gpu 0

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/ASTA/train.py:136: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than w

## Phase 3: Test Trained Weights

In [ ]:
#@title 10. Run Inference with Trained Weights
!python infer.py \
  --model asta \
  --weights /content/drive/MyDrive/ASTA_Results/weights/asta.pth \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --result_dir /content/drive/MyDrive/ASTA_Results/trained_results/

In [ ]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/ASTA_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/